# Phishing Email Detection System
**IBM SkillsBuild Cyber Security BootCamp 2026 | Beginner Python Project**

## Project overview
This beginner-friendly project checks the text of an email and flags common phishing warning signs. It uses transparent, rule-based checks rather than a trained machine-learning model. It is intended for learning and awareness—not as a replacement for an organization's email security gateway or human review.

### What it checks
- Urgent or threatening language
- Requests for passwords, one-time passcodes (OTPs), or account details
- Suspicious links, including URL shorteners and IP-address-based links
- Common attachment warning signs
- Mismatch between a displayed sender name and sender address (when supplied)

**Safety note:** Use only sample or authorized email text. Do not paste passwords, OTPs, private messages, or confidential company information into this notebook.

## 1. Setup
The project uses only Python's standard library, so no external packages are required.

1. Install Python 3.10 or later.
2. Open this notebook in Jupyter Notebook, JupyterLab, or Google Colab.
3. Run the cells from top to bottom.

In [ ]:
import re
from urllib.parse import urlparse
from collections import Counter

## 2. Detection rules
Each matching indicator adds points to a simple risk score. The score is capped at 100.

This is a demonstration scoring system, not a statistically validated probability. A legitimate email can contain warning words, and a carefully written phishing email may avoid these indicators.

In [ ]:
URGENT_TERMS = [
    "urgent", "immediately", "act now", "verify now",
    "account suspended", "account will be closed", "final warning",
    "within 24 hours", "unusual activity"
]

CREDENTIAL_TERMS = [
    "password", "one-time passcode", "otp", "login details",
    "bank details", "credit card", "security code"
]

PAYMENT_TERMS = [
    "wire transfer", "gift card", "payment required",
    "update your payment", "send money"
]

SHORTENERS = {
    "bit.ly", "tinyurl.com", "t.co", "is.gd", "cutt.ly", "ow.ly"
}

RISKY_EXTENSIONS = (
    ".exe", ".scr", ".js", ".vbs", ".bat", ".cmd", ".zip", ".iso"
)

def find_urls(text):
    # Simple URL extraction suitable for a classroom demonstration.
    return re.findall(r"https?://[^\\s<>\\\"']+", text, flags=re.IGNORECASE)

def analyze_email(subject, body, sender_name="", sender_address="", attachments=None):
    attachments = attachments or []
    combined = f"{subject} {body}".lower()
    indicators = []
    score = 0

    urgent_hits = [term for term in URGENT_TERMS if term in combined]
    if urgent_hits:
        score += min(20, 10 + 2 * (len(urgent_hits) - 1))
        indicators.append("Urgent or threatening language: " + ", ".join(urgent_hits))

    credential_hits = [term for term in CREDENTIAL_TERMS if term in combined]
    if credential_hits:
        score += min(30, 15 + 3 * (len(credential_hits) - 1))
        indicators.append("Sensitive information requested: " + ", ".join(credential_hits))

    payment_hits = [term for term in PAYMENT_TERMS if term in combined]
    if payment_hits:
        score += 15
        indicators.append("Payment-related pressure: " + ", ".join(payment_hits))

    urls = find_urls(subject + " " + body)
    for url in urls:
        parsed = urlparse(url)
        host = (parsed.hostname or "").lower()
        if host in SHORTENERS:
            score += 15
            indicators.append(f"URL shortener detected: {host}")
        # A URL using an IP address instead of a recognizable domain can be suspicious.
        if re.fullmatch(r"\\d{1,3}(?:\\.\\d{1,3}){3}", host):
            score += 20
            indicators.append(f"URL uses an IP address: {host}")
        if parsed.scheme.lower() != "https":
            score += 5
            indicators.append("Link does not use HTTPS")

    risky_files = [name for name in attachments
                   if name.lower().endswith(RISKY_EXTENSIONS)]
    if risky_files:
        score += 20
        indicators.append("Potentially risky attachment type: " + ", ".join(risky_files))

    if sender_name and sender_address:
        # A deliberately simple check: compare the sender-name words with the email domain.
        domain = sender_address.split("@")[-1].lower() if "@" in sender_address else ""
        sender_words = [w for w in re.findall(r"[a-z0-9]+", sender_name.lower()) if len(w) > 3]
        if domain and sender_words and not any(w in domain for w in sender_words):
            score += 10
            indicators.append("Sender display name may not match the email domain")

    score = min(score, 100)
    if score >= 50:
        label = "HIGH RISK — likely phishing indicators"
    elif score >= 25:
        label = "CAUTION — review carefully"
    else:
        label = "LOWER RISK — no strong indicators found"

    if not indicators:
        indicators = ["No configured warning indicators were detected. This does not prove the email is safe."]

    return {
        "risk_score": score,
        "classification": label,
        "indicators": indicators,
        "urls_found": urls
    }

## 3. Try the detector
The following example is fictional. It is included only to demonstrate how the rules work.

In [ ]:
sample_subject = "URGENT: Verify your account immediately"
sample_body = (
    "Your account will be closed within 24 hours. "
    "Confirm your password at http://bit.ly/example"
)
sample_result = analyze_email(
    sample_subject,
    sample_body,
    sender_name="Example Bank Support",
    sender_address="alerts@unrelated-example.net",
    attachments=["account_update.exe"]
)

sample_result

## 4. Test with a routine email

In [ ]:
routine_subject = "Reminder: project meeting tomorrow"
routine_body = (
    "Hello team, our project meeting is scheduled for 10:00 AM tomorrow. "
    "Please bring your progress notes."
)

routine_result = analyze_email(routine_subject, routine_body)
routine_result

## 5. Try your own safe sample
Replace the example text below with fictional or publicly available training text. Do not enter real credentials or confidential information.

In [ ]:
subject = "Package delivery update"
body = "Your parcel is ready. View the tracking details on the courier's official website."
result = analyze_email(subject, body)
print("Classification:", result["classification"])
print("Risk score:", result["risk_score"], "/ 100")
print("Warning indicators:")
for item in result["indicators"]:
    print("-", item)

## 6. Basic test cases
These checks help confirm that the function returns a result and applies expected labels to example inputs.

In [ ]:
tests = [
    ("Routine note", "Meeting notes are attached for tomorrow.", "LOWER RISK"),
    ("Credential request", "Verify your password immediately.", "HIGH RISK"),
    ("Shortened link", "Open https://bit.ly/example to review.", "CAUTION"),
]

for name, body_text, expected_prefix in tests:
    result = analyze_email(name, body_text)
    print(name, "=>", result["classification"], "| score:", result["risk_score"])

## 7. Limitations and responsible use
- This is a rule-based educational prototype, not a production security product.
- It does not inspect email headers, authenticate senders, follow links, scan files, or use threat-intelligence feeds.
- Risk scores are illustrative and are not probabilities.
- Legitimate emails may be flagged (false positives), and phishing emails may be missed (false negatives).
- Never open suspicious links or attachments just to test them. Report suspicious messages through your institution's approved process.
- Do not use this tool to make automated decisions about people or to claim that an email is definitively safe.

## 8. Further improvements
Possible future enhancements include a labeled training dataset, precision/recall evaluation, safe URL reputation lookups, email-header analysis, and a simple user interface—subject to privacy and security review.

## Dataset note
The notebook contains fictional examples and does not require downloading a dataset. For optional study of email spam classification, see the UCI Spambase dataset:
https://archive.ics.uci.edu/dataset/94/spambase

Spambase is a general spam dataset, not a dedicated phishing dataset. It should not be described as a phishing-labeled dataset.